In [0]:
-- We create a table to obtain the minimum, maximum and average values for the process variables in the silver table. First, we run a CTE to generate values that will be used to show on the gold table and to perform calculations such as the temperature difference between the maximum and minimum:


CREATE OR REPLACE TABLE databricks_cnc_machine_project.min_max_avg_values_gold AS

-- CTE:

(WITH min_max_avg_values AS

(SELECT

    -- Minimum values in the silver layer:
    MIN(`air_temperature_k`) as min_air_temp_k,
    MIN(`process_temperature_k`) as min_process_temp_k,
    MIN(`power_watts`) as min_power_watts,
    MIN(`torque_nm`) as min_torque_nm,
    MIN(`rotational_speed_rpm`) as min_rot_speed_rpm,

    -- Maximum values in the silver layer:
    MAX(`air_temperature_k`) as max_air_temp_k,
    MAX(`process_temperature_k`) as max_process_temp_k,
    MAX(`power_watts`) as max_power_watts,
    MAX(`torque_nm`) as max_torque_nm,
    MAX(`rotational_speed_rpm`) as max_rot_speed_rpm,

    -- Average values in the silver layer:
    ROUND(AVG(`air_temperature_k`), 2) as avg_air_temp_k,
    ROUND(AVG(`process_temperature_k`), 2) as avg_process_temp_k,
    ROUND(AVG(`power_watts`), 2) as avg_power_watts,
    ROUND(AVG(`torque_nm`), 2) as avg_torque_nm,
    AVG(`rotational_speed_rpm`)::INT as avg_rot_speed_rpm

FROM databricks_cnc_machine_project.cnc_machine_silver_layer)

-- Query that uses the CTE values:

SELECT
  *,
  ROUND(max_air_temp_k - min_air_temp_k, 2) AS air_temp_diff_k,
  ROUND(max_process_temp_k - min_process_temp_k, 2) AS process_temp_diff_k,
  ROUND(max_power_watts - min_power_watts, 2) AS power_diff_watts,
  ROUND(max_torque_nm - min_torque_nm, 2) AS torque_diff_nm,
  (max_rot_speed_rpm - min_rot_speed_rpm) AS rot_speed_diff_rpm,

  -- Timestamp for better tracking of the gold table:
  CURRENT_TIMESTAMP() AS _gold_processed_timestamp   
FROM min_max_avg_values)


------------------------------------------------------------------------------
-- After the table has been created, a query is run to verify it:

SELECT *
FROM databricks_cnc_machine_project.min_max_avg_values_gold



In [0]:
-- We created a table similar to the silver layer, adding a column to identify how many failure modes might have occurred, as well as another column that serves as a semantic layer to transform technical indicators into descriptive labels. 

-- This makes it easier to interpret the data in BI tools, allowing end users to quickly identify complex failures without needing to understand the underlying logic of the counters.

CREATE OR REPLACE TABLE databricks_cnc_machine_project.cnc_machine_gold_layer AS

(SELECT 
    *,
   
    -- This step is for calculations and logical filters:
    (tool_wear_failure::INT + heat_dissipation_failure::INT + 
     power_failure::INT + overstrain_failure::INT) AS failure_modes_count,
    
    -- Semantic Layer using Case logic for better visualization and business users:
    CASE 
        WHEN machine_failure = False THEN 'No Failure'
        WHEN (tool_wear_failure::INT + heat_dissipation_failure::INT + 
              power_failure::INT + overstrain_failure::INT) = 1 THEN 'Single Mode Failure'
        WHEN (tool_wear_failure::INT + heat_dissipation_failure::INT + 
              power_failure::INT + overstrain_failure::INT) > 1 THEN 'Multi-Mode Failure'
        -- Else if any unexpected result appears
        ELSE 'Review Required'
    END AS failure_severity_tag,

    -- Timestamp for better tracking of the gold table:
    CURRENT_TIMESTAMP() AS _gold_processed_timestamp 
    
FROM databricks_cnc_machine_project.cnc_machine_silver_layer)


------------------------------------------------------------------------------
-- After the table has been created, a query is run to verify it:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_gold_layer



In [0]:
-- We create a table with a summary of failed and succesful operations. Since failure values are Boolean, when you apply the simplified CAST function (using ::), the value True returns a 1, False returns a 0, which simplifies and optimizes the count of failure mode occurrences:

CREATE OR REPLACE TABLE databricks_cnc_machine_project.summary_total_operations_gold AS

(SELECT

-- Total operations (with and without machine failure)
COUNT(*) AS total_operations,

-- Total succesful operations (Machine Failure = False)
SUM(CASE WHEN machine_failure = False THEN 1 ELSE 0 END) AS total_successful_operations,

-- Total failed operations (Machine Failure = True)
SUM(machine_failure::INT) AS total_machine_failures,

-- Total failure rate percentage
ROUND((SUM(machine_failure::INT) * 100.0) / COUNT(*), 2) AS total_failure_rate_percentage,

-- Total failures by failure mode:
SUM(`tool_wear_failure`::INT) AS total_tool_wear_failures,
ROUND((SUM(`tool_wear_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_tool_wear_failures,

SUM(`heat_dissipation_failure`::INT) AS total_heat_dissipation_failures,
ROUND((SUM(`heat_dissipation_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_heat_dissipation_failures,

SUM(`power_failure`::INT) AS total_power_failures,
ROUND((SUM(`power_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_power_failures,

SUM(`overstrain_failure`::INT) AS total_overstrain_failures,
ROUND((SUM(`overstrain_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_overstrain_failures,

SUM(`random_failure`::INT) AS total_random_failures,
ROUND((SUM(`random_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_random_failures,

-- Timestamp for better tracking of the gold table:
CURRENT_TIMESTAMP() AS _gold_processed_timestamp 

FROM databricks_cnc_machine_project.cnc_machine_silver_layer)


------------------------------------------------------------------------------
-- After the table has been created, a query is run to verify it:

SELECT *
FROM databricks_cnc_machine_project.summary_total_operations_gold


In [0]:
-- In this step, we create a table with a summary, per product type, of failed and succesful operations:

CREATE OR REPLACE TABLE databricks_cnc_machine_project.summary_operations_per_product_type_gold AS

(SELECT
product_type,

-- Total operations (with and without machine failure)
COUNT(*) AS total_operations,

-- Total succesful operations (Machine Failure = False)
SUM(CASE WHEN machine_failure = False THEN 1 ELSE 0 END) AS total_successful_operations,

-- Total failed operations (Machine Failure = True)
SUM(machine_failure::INT) AS total_machine_failures,

-- Total failure percentage rate (%)
ROUND((SUM(machine_failure::INT) * 100.0) / COUNT(*), 2) AS total_failure_rate_percentage,

-- Tool Wear Failures
SUM(tool_wear_failure::INT) AS total_tool_wear_failures,
ROUND((SUM(tool_wear_failure::INT) * 100.0) / COUNT(*), 2) AS percent_tool_wear_failures,

-- Heat Dissipation Failures
SUM(heat_dissipation_failure::INT) AS total_heat_dissipation_failures,
ROUND((SUM(heat_dissipation_failure::INT) * 100.0) / COUNT(*), 2) AS percent_heat_dissipation_failures,

-- Power Failures
SUM(power_failure::INT) AS total_power_failures,
ROUND((SUM(power_failure::INT) * 100.0) / COUNT(*), 2) AS percent_power_failures,

-- Overstrain Failures
SUM(overstrain_failure::INT) AS total_overstrain_failures,
ROUND((SUM(overstrain_failure::INT) * 100.0) / COUNT(*), 2) AS percent_overstrain_failures,

-- Random Failures
SUM(random_failure::INT) AS total_random_failures,
ROUND((SUM(random_failure::INT) * 100.0) / COUNT(*), 2) AS percent_random_failures,

-- Timestamp for better tracking of the gold table:
CURRENT_TIMESTAMP() AS _gold_processed_timestamp

FROM databricks_cnc_machine_project.cnc_machine_silver_layer
GROUP BY product_type
ORDER BY 
    CASE product_type
        WHEN 'L' THEN 1
        WHEN 'M' THEN 2
        WHEN 'H' THEN 3
        ELSE 4 
    END)


------------------------------------------------------------------------------
-- After the table has been created, a query is run to verify it:

SELECT *
FROM databricks_cnc_machine_project.summary_operations_per_product_type_gold


In [0]:
-- We create a table containing information about rows where the tool is changed at the end of the operation. This allows us to determine the tool's lifetime and whether the tool was changed due to a failure:

CREATE OR REPLACE TABLE databricks_cnc_machine_project.tool_lifecycle_gold AS 

-- CTE to identify rows where the tool is changed (tool_wear_min = 0):

(WITH table_toolwearmin_0 AS

(SELECT
*
FROM databricks_cnc_machine_project.cnc_machine_silver_layer
WHERE tool_wear_min = 0 AND silver_id > 1)


-- Query using the previous CTE and selecting the rows before the tool is changed:

SELECT
*,
(machine_failure::INT +
tool_wear_failure::INT + 
heat_dissipation_failure::INT + 
power_failure::INT + 
overstrain_failure::INT) AS total_failures_at_tool_change,
-- Timestamp for better tracking of the gold table:
CURRENT_TIMESTAMP() AS _gold_processed_timestamp
FROM databricks_cnc_machine_project.cnc_machine_silver_layer
WHERE silver_id IN (SELECT (silver_id - 1) FROM table_toolwearmin_0))


------------------------------------------------------------------------------
-- After the table has been created, a query is run to verify it:

SELECT *
FROM databricks_cnc_machine_project.tool_lifecycle_gold
